In [0]:
# Configrations
source_dir = "/Volumes/external-catalog/bronze/incremental_load/orders_data/source/"
archive_dir = "/Volumes/external-catalog/bronze/incremental_load/orders_data/archive/"
orders_bronze_table = "`external-catalog`.bronze.orders_bronze"
orders_bronze_error_table = "`external-catalog`.bronze.orders_bronze_error"

print(f"Processing orders data from : {source_dir}")
print(f"Archiving processed data to : {archive_dir}")
print(f"Writing processed data to : {orders_bronze_table}")
print(f"Writing error data to : {orders_bronze_error_table}")

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import *
from datetime import datetime
import json

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("order_amount", DecimalType(10,2), False),
    StructField("currency", StringType(), False),
    StructField("payment_method", StringType(), False),
    StructField("shipping_address", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("created_timestamp", TimestampType(), False)
])

print("Schema defined for orders data")

In [0]:
try:
    df_orders = spark.read.schema(orders_schema).csv(source_dir, header=True, dateFormat="yyyy-MM-dd")
    print("Orders data read from source")
    df_orders = df_orders.withColumn("proceses_timestamp", f.current_timestamp()) \
        .withColumn("batch_id", f.lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) \
        .withColumn("source_system", f.lit("ecommerce_orders"))

    df_orders.printSchema()
    
    #Data Quality Checks
    total_records = df_orders.count()
    null_orders_ids = df_orders.filter(f.col("order_id").isNull()).count()
    null_customer_ids = df_orders.filter(f.col("customer_id").isNull()).count()
    invalid_amounts = df_orders.filter(f.col("order_amount") <= 0).count()
    
    print(f"Total records in orders data : {total_records}")
    print(f"Null order ids : {null_orders_ids}")
    print(f"Null customer ids : {null_customer_ids}")
    print(f"Invalid amounts : {invalid_amounts}")
    
    # Filtering out the invalid records as mentioned above
    df_orders_valid = df_orders.filter(f.col("order_id").isNotNull() & f.col("customer_id").isNotNull() & (f.col("order_amount") > 0))
    

    df_orders_invalid = df_orders.filter(f.col("order_id").isNull() | f.col("customer_id").isNull() | (f.col("order_amount") <= 0))

    valid_records = df_orders.count()
    invalid_records = df_orders_invalid.count()

    print(f"Valid records in orders data : {valid_records}")
    print(f"Invalid records in orders data : {invalid_records}")
    
except Exception as e:
    print(f"Error while reading orders data from source : {e}")
    raise
    

In [0]:
#Writing valid data to the staging table

try:
    df_orders_valid.write.format("delta").mode("overwrite").saveAsTable(orders_bronze_table)
    print(f"Valid orders data written to {orders_bronze_table}")

    if invalid_records > 0:
        df_orders_invalid.write.format("delta").mode("overwrite").saveAsTable(orders_bronze_error_table)
        print(f"Invalid orders data written to {orders_bronze_error_table}")
except Exception as e:
    print(f"Error while writing orders data to {orders_bronze_table} : {e}")
    raise

In [0]:
# Archive the processed data
try:
    files = dbutils.fs.ls(source_dir)

    archived_count = 0
    for file in files:
        if file.name.endswith(".csv"):
            src_path = file.path
            archive_path = archive_dir + file.name

            #Move the file to arhive path

            dbutils.fs.mv(src_path, archive_path)
            archived_count += 1

    print(f"Archived {archived_count} files to {archive_dir}")
except Exception as e:
    print(f"Error while archiving processed data : {e}")
    raise

In [0]:
# Log processing summary
processing_summary = {
    "task": "order_bronze_load",
    "timestamp": datetime.now().isoformat(),
    "total_records": total_records,
    "valid_records": valid_records,
    "invalid_records": invalid_records,
    "archived_files": archived_count,
    "status": "SUCCESS" if invalid_records == 0 else "SUCCESS_WITH_WARNINGS"
}

print(f"Processing summary : {processing_summary}")
print(json.dumps(processing_summary, indent=2))

summary_df = spark.createDataFrame([processing_summary])
summary_df.write.mode("append").format("delta").saveAsTable("`external-catalog`.default.processing_control_table")